In [39]:
# 1. import pandas
import pandas as pd

# 2. load 2024 FARS accident data
accident = pd.read_csv("data/raw/FARS2024NationalCSV/accident.csv")

# 3. original dataset
accident.shape

(36297, 80)

### Data Process for Q1: time of crashes

In [40]:
# 4. Keep variables needed for Q1
q1 = accident[['ST_CASE','DAY_WEEK','DAY_WEEKNAME','HOUR','HOURNAME','FATALS']].copy()

# 5. Check for duplicate crashes
q1['ST_CASE'].duplicated().sum()

# 6. day of week - we want sunday as "1"
q1[['DAY_WEEK', 'DAY_WEEKNAME']].drop_duplicates().sort_values('DAY_WEEK')

,DAY_WEEK,DAY_WEEKNAME
31,1,Sunday
0,2,Monday
1,3,Tuesday
3,4,Wednesday
5,5,Thursday
8,6,Friday
11,7,Saturday


In [41]:
# 7. number of crashes by day
q1['DAY_WEEKNAME'].value_counts()

DAY_WEEKNAME
Saturday     6218
Sunday       5597
Friday       5588
Thursday     4832
Monday       4766
Wednesday    4753
Tuesday      4543
Name: count, dtype: int64

In [42]:
#8. Sort crashes by day of week
q1 = q1.sort_values('DAY_WEEK').reset_index(drop=True)
q1.head()

,ST_CASE,DAY_WEEK,DAY_WEEKNAME,HOUR,HOURNAME,FATALS
0,360697,1,Sunday,17,5:00pm-5:59pm,1
1,370663,1,Sunday,17,5:00pm-5:59pm,1
2,370667,1,Sunday,21,9:00pm-9:59pm,2
3,370668,1,Sunday,16,4:00pm-4:59pm,1
4,190041,1,Sunday,12,12:00pm-12:59pm,1


In [53]:
# Fatal crashes by day of week and hour
q1_day_hour = (
    q1.groupby(['DAY_WEEK', 'DAY_WEEKNAME', 'HOUR', 'HOURNAME'])
      .agg(COUNT_ST_CASE=('ST_CASE', 'count'))
      .reset_index()
      .sort_values('COUNT_ST_CASE', ascending=False)
)

q1_day_hour.head(2)

,DAY_WEEK,DAY_WEEKNAME,HOUR,HOURNAME,COUNT_ST_CASE
170,7,Saturday,20,8:00pm-8:59pm,420
173,7,Saturday,23,11:00pm-11:59pm,414


In [44]:
#9. save  Q1 processed data
q1.to_csv('data/processed/Q1.csv',index=False)

### Data process for Q2: Fatal Crash Rate by State 

In [64]:
# Load 2024 FARS accident data
pop = pd.read_excel("data/raw/2024POP.xlsx",header=3)

# Keep rows 10–60 from the Excel file
# Excel rows 10–60 correspond to pandas rows 5–55
population = pop.iloc[5:56, [0, 6]].copy()

# Rename columns
population.columns = ["State", "Population_2024"]

population.head()

,State,Population_2024
5,.Alabama,5157699.0
6,.Alaska,740133.0
7,.Arizona,7582384.0
8,.Arkansas,3088354.0
9,.California,39431263.0


In [67]:
# Remove "." from state names
population["State"] = population["State"].str.lstrip(".")

# Convert population to integer
population["Population_2024"] = population["Population_2024"].astype(int)

population.head()

,State,Population_2024
5,Alabama,5157699
6,Alaska,740133
7,Arizona,7582384
8,Arkansas,3088354
9,California,39431263


In [70]:
#merge
fars_pop = df.merge(
    population,
    left_on="STATENAME",
    right_on="State",
    how="left"
)
fars_pop["Population_2024"].isna().sum()

np.int64(0)

In [72]:
q2 = (
    fars_pop
    .groupby(["STATENAME", "Population_2024"])
    .size()
    .reset_index(name="Fatal_Crashes")
)
q2["Fatal_Crashes_per_100k"] = (
    q2["Fatal_Crashes"] /
    q2["Population_2024"] *
    100000
)
q2.head()


,STATENAME,Population_2024,Fatal_Crashes,Fatal_Crashes_per_100k
0,Alabama,5157699,895,17.352699
1,Alaska,740133,63,8.511984
2,Arizona,7582384,1118,14.744703
3,Arkansas,3088354,547,17.711700
4,California,39431263,3583,9.086699


In [77]:
q2.sort_values(
    "Fatal_Crashes_per_100k",
    ascending=False
).head(3)

,STATENAME,Population_2024,Fatal_Crashes,Fatal_Crashes_per_100k
24,Mississippi,2943045,678,23.037364
31,New Mexico,2130256,378,17.744346
3,Arkansas,3088354,547,17.711700


In [76]:
#save data
q2.to_csv(
    "data/processed/Q2.csv",
    index=False
)

### Q3: How have fatal crash patterns changed over time?

In [8]:
accident_2024 = pd.read_csv('data/raw/FARS2024NationalCSV/accident.csv')
print("2024:", accident_2024.shape)

2024: (36297, 80)


In [9]:
accident_2023 = pd.read_csv('data/raw/FARS2023NationalCSV/accident.csv')
print("2023:", accident_2023.shape)

2023: (37769, 80)


In [10]:
accident_2022 = pd.read_csv('data/raw/FARS2022NationalCSV/accident.csv')
print("2022:", accident_2022.shape)

2022: (39422, 80)


In [11]:
accident_2021 = pd.read_csv('data/raw/FARS2021NationalCSV/accident.csv')
print("2021:", accident_2021.shape)

2021: (39785, 80)


In [12]:
accident_2020 = pd.read_csv('data/raw/FARS2020NationalCSV/accident.csv',encoding='latin1')
print("2020:", accident_2020.shape)

2020: (35935, 81)


In [13]:
print("2020 extra columns:")
print(set(accident_2020.columns) - set(accident_2021.columns))

print("\n2021 extra columns:")
print(set(accident_2021.columns) - set(accident_2020.columns))

2020 extra columns:
{'DRUNK_DR'}

2021 extra columns:
set()


In [14]:
accident_2020 = accident_2020.drop(columns=['DRUNK_DR'])
print("2020:", accident_2020.shape)

2020: (35935, 80)


In [15]:
accident_2020 = accident_2020[accident_2021.columns]
print(accident_2020.columns.equals(accident_2021.columns))

True


In [16]:
q3 = pd.concat([accident_2020,accident_2021,accident_2022,accident_2023,
        accident_2024],ignore_index=True)
print("Q3:", q3.shape)

Q3: (189208, 80)


In [17]:
# only keep the useful variables 
q3 = q3[['YEAR', 'ST_CASE', 'DAY_WEEKNAME', 'HOUR', 'HOURNAME']]
print(q3.shape)
print(q3.columns.tolist())

(189208, 5)
['YEAR', 'ST_CASE', 'DAY_WEEKNAME', 'HOUR', 'HOURNAME']


In [21]:
year_total = q3.groupby('YEAR')['ST_CASE'].count()
print(year_total)

YEAR
2020    35935
2021    39785
2022    39422
2023    37769
2024    36297
Name: ST_CASE, dtype: int64


In [22]:
hourly_count = (
    q3.groupby(['YEAR', 'HOUR'])['ST_CASE']
      .count()
      .reset_index(name='FATAL_CRASHES')
)

print(hourly_count.head(10))

   YEAR  HOUR  FATAL_CRASHES
0  2020     0           1483
1  2020     1           1254
2  2020     2           1161
3  2020     3            916
4  2020     4            939
5  2020     5           1039
6  2020     6           1276
7  2020     7           1062
8  2020     8            874
9  2020     9            916


In [24]:
hourly_count['PERCENT_OF_YEAR'] = (
    hourly_count['FATAL_CRASHES']
    / hourly_count['YEAR'].map(year_total)
    * 100)
print(hourly_count.head(10))

   YEAR  HOUR  FATAL_CRASHES  PERCENT_OF_YEAR
0  2020     0           1483         4.126896
1  2020     1           1254         3.489634
2  2020     2           1161         3.230833
3  2020     3            916         2.549047
4  2020     4            939         2.613051
5  2020     5           1039         2.891332
6  2020     6           1276         3.550856
7  2020     7           1062         2.955336
8  2020     8            874         2.432169
9  2020     9            916         2.549047


In [25]:
hourly_count.groupby('YEAR')['PERCENT_OF_YEAR'].sum()

YEAR
2020    100.0
2021    100.0
2022    100.0
2023    100.0
2024    100.0
Name: PERCENT_OF_YEAR, dtype: float64

In [26]:
hourly_count.loc[
    hourly_count.groupby('YEAR')['PERCENT_OF_YEAR'].idxmax()
]

,YEAR,HOUR,FATAL_CRASHES,PERCENT_OF_YEAR
18,2020,18,2198,6.116599
43,2021,18,2397,6.024884
71,2022,21,2336,5.925625
95,2023,20,2328,6.163785
120,2024,20,2195,6.047332


In [27]:
hourly_count[hourly_count['HOUR'] == 99]

,YEAR,HOUR,FATAL_CRASHES,PERCENT_OF_YEAR
24,2020,99,305,0.848755
49,2021,99,300,0.754053
74,2022,99,295,0.748313
99,2023,99,271,0.717520
124,2024,99,253,0.697027


In [28]:
hour_label = q3[['YEAR', 'HOUR', 'HOURNAME']].drop_duplicates()

hourly_count = hourly_count.merge(
    hour_label,
    on=['YEAR', 'HOUR'],
    how='left'
)

print(hourly_count.head())

   YEAR  HOUR  FATAL_CRASHES  PERCENT_OF_YEAR       HOURNAME
0  2020     0           1483         4.126896  0:00am-0:59am
1  2020     1           1254         3.489634  1:00am-1:59am
2  2020     2           1161         3.230833  2:00am-2:59am
3  2020     3            916         2.549047  3:00am-3:59am
4  2020     4            939         2.613051  4:00am-4:59am


In [35]:
peak_hour = (
    hourly_count
    .loc[hourly_count.groupby('YEAR')['PERCENT_OF_YEAR'].idxmax()]
    .reset_index(drop=True)
)

print(peak_hour)

   YEAR  HOUR  FATAL_CRASHES  PERCENT_OF_YEAR       HOURNAME
0  2020    18           2198         6.116599  6:00pm-6:59pm
1  2021    18           2397         6.024884  6:00pm-6:59pm
2  2022    21           2336         5.925625  9:00pm-9:59pm
3  2023    20           2328         6.163785  8:00pm-8:59pm
4  2024    20           2195         6.047332  8:00pm-8:59pm


In [36]:
#save data for q3
peak_hour.to_csv('data/processed/Q3_peak_hour_2020_2024.csv',index=False)